In [1]:
import torch
import torch.nn as nn

In [2]:
if torch.cuda.is_available():
    device="cuda"
else:
    device="cpu"

In [3]:
class MultiHeadedAttention(nn.Module):
    def __init__(self,d_model,n_qk,n_v,n_head):
        super().__init__()
        self.n_head=n_head
        self.n_qk=n_qk
        self.n_v=n_v
        self.softmax=nn.Softmax(dim=-1)
        self.Wq=nn.Linear(d_model,n_qk*n_head)
        self.Wk=nn.Linear(d_model,n_qk*n_head)
        self.Wv=nn.Linear(d_model,n_v*n_head)
        self.Wo=nn.Linear(n_v*n_head,d_model)

    def forward(self,Q_input,K_input,V_input,mask=None):
      #First we are taking all the Queuries,Keys and Value matrices into one matrix each
        Q=self.Wq(Q_input)     #Current shape:(batch,n_seq,n_qk*n_head)
        K=self.Wk(K_input)     #Current shape:(batch,n_seq,n_qk*n_head)
        V=self.Wv(V_input)     #Current shape:(batch,n_seq,n_v*n_head)

     #Now we divide in into different heads
        Q=Q.view(Q.size(0),Q.size(1),self.n_head,self.n_qk)  #Current shape:(batch,n_seq,n_head,n_qk)
        K=K.view(K.size(0),K.size(1),self.n_head,self.n_qk)  #Current shape:(batch,n_seq,n_head,n_qk)
        V=V.view(V.size(0),V.size(1),self.n_head,self.n_v)  #Current shape:(batch,n_seq,n_head,n_v)
        
        Q=Q.transpose(1,2)   #Current shape:(batch,n_head,n_seq,n_qk)
        K=K.transpose(1,2)   #Current shape:(batch,n_head,n_seq,n_qk)
        V=V.transpose(1,2)   #Current shape:(batch,n_head,n_seq,n_v)

    #Now we apply the Vsoftmax(QK(Transpose)/root(d_k)) 
        dot_product=(Q@K.transpose(2,3))/((self.n_qk)**0.5)     #Current shape:(batch,n_head,n_seq,n_seq)
        if mask is not None:
            dot_product=dot_product.masked_fill(mask,float("-inf"))
        value_output=self.softmax(dot_product)@V                #Current shape:(batch,n_head,n_seq,n_v)


    #Now we concat back the value_output together
        value_concat=(value_output.transpose(1,2)).contiguous().view(value_output.size(0),value_output.size(2),(self.n_head)*(self.n_v)) #Current shape:(batch,n_seq,n_head*n_v)

    #We now make it compatible for adding to the input matrix and then return it 
        final_output=self.Wo(value_concat)        #Current shape:(batch,n_seq,d_model)
        return final_output

In [4]:
class RelativeMultiHeadedAttention(nn.Module):
    def __init__(self,d_model,n_qk,n_v,n_head,clipping_length):
        super().__init__()
        self.n_head=n_head
        self.n_qk=n_qk
        self.n_v=n_v
        self.softmax=nn.Softmax(dim=-1)
        self.Wq=nn.Linear(d_model,n_qk*n_head)
        self.Wk=nn.Linear(d_model,n_qk*n_head)
        self.Wv=nn.Linear(d_model,n_v*n_head)
        self.Wo=nn.Linear(n_v*n_head,d_model)
        self.clipping_k=nn.Embedding(2*clipping_length+1,n_qk)
        self.clipping_v=nn.Embedding(2*clipping_length+1,n_v)
        self.clipping_length=clipping_length

    def forward(self,Q_input,K_input,V_input,mask=None):
      #First we are taking all the Queuries,Keys and Value matrices into one matrix each
        Q=self.Wq(Q_input)     #Current shape:(batch,n_seq,n_qk*n_head)
        K=self.Wk(K_input)     #Current shape:(batch,n_seq,n_qk*n_head)
        V=self.Wv(V_input)     #Current shape:(batch,n_seq,n_v*n_head)

     #Now we divide in into different heads
        Q=Q.view(Q.size(0),Q.size(1),self.n_head,self.n_qk)  #Current shape:(batch,n_seq,n_head,n_qk)
        K=K.view(K.size(0),K.size(1),self.n_head,self.n_qk)  #Current shape:(batch,n_seq,n_head,n_qk)
        V=V.view(V.size(0),V.size(1),self.n_head,self.n_v)  #Current shape:(batch,n_seq,n_head,n_v)
        
        Q=Q.transpose(1,2)   #Current shape:(batch,n_head,n_seq,n_qk)
        K=K.transpose(1,2)   #Current shape:(batch,n_head,n_seq,n_qk)
        V=V.transpose(1,2)   #Current shape:(batch,n_head,n_seq,n_v)
      
        position_absolute=torch.arange(0,Q.size(2)).to(Q.device) #Initializing a 1-d tensor with all the relative distances possible
        row_position=position_absolute.unsqueeze(0)              #Making a row tensor
        column_position=position_absolute.unsqueeze(1)           #Making a column tensor
        relative_position=row_position-column_position           #Using broadcasting to calculate the grid of relative positions of each combination possible
        clipped_relative=torch.clamp(relative_position,-self.clipping_length,self.clipping_length) #Clipping the grid of relative positions
        clipped_relative_positive=clipped_relative+self.clipping_length     #Making the grid go from 0 to 2k to make the grid embedding indexing easier
        a_k=self.clipping_k(clipped_relative_positive)                      #Embedding the grid such that each distance in the grid gets its own tensor
        a_v=self.clipping_v(clipped_relative_positive)
        Q_fitted=Q.unsqueeze(3)                      #To make dot product broadcast with a_k, Current Shape:(batch,head,n_seq,1,n_qk)
        a_k_fitted=a_k.unsqueeze(0).unsqueeze(0)     #To make dot product broadcast with Q, Current Shape:(1,1,n_seq,n_seq,n_qk)
    #Now we apply the Vsoftmax((QK(Transpose)+Q(a_k))/root(d_k))
        dot_product_a_k=(Q_fitted*a_k_fitted).sum(axis=-1)/((self.n_qk)**0.5)
        dot_product_K=(Q@K.transpose(2,3))/((self.n_qk)**0.5)     #Current shape:(batch,n_head,n_seq,n_seq)
        final_dot_product=dot_product_a_k+dot_product_K
        if mask is not None:
            final_dot_product=final_dot_product.masked_fill(mask,float("-inf"))
        attention_score=self.softmax(final_dot_product)
        value_output_V=attention_score@(V)                #Current shape:(batch,n_head,n_seq,n_v)
        attention_score_fitted=attention_score.unsqueeze(-1)
        a_v_fitted=a_v.unsqueeze(0).unsqueeze(0)
        value_output_a_v=(attention_score_fitted*a_v_fitted).sum(dim=3)
        value_output=value_output_V + value_output_a_v
    #Now we concat back the value_output together
        value_concat=(value_output.transpose(1,2)).contiguous().view(value_output.size(0),value_output.size(2),(self.n_head)*(self.n_v)) #Current shape:(batch,n_seq,n_head*n_v)

    #We now make it compatible for adding to the input matrix and then return it 
        final_output=self.Wo(value_concat)        #Current shape:(batch,n_seq,d_model)
        return final_output

In [5]:
#This the fully connected layer after the attention block in both the decoder and the encoder layers
class FeedForward(nn.Module):
    def __init__(self,d_model,n_neurons):
      super().__init__()
      self.feedforward=nn.Sequential(nn.Linear(d_model,n_neurons),
                                    nn.ReLU(),
                                    nn.Linear(n_neurons,d_model))

    def forward(self,X):
        output=self.feedforward(X)
        return output

In [6]:
#The encoder layer consists of relative position attention block with a residual connection and normalization followed by a fully connected layer with a 
#residual connection and normalization
class EncoderLayer(nn.Module):
    def __init__(self,d_model,n_qk,n_v,n_head,clipping_length,n_neurons):
        super().__init__()
        self.RelativeAttentionBlock=RelativeMultiHeadedAttention(d_model,n_qk,n_v,n_head,clipping_length)
        self.norm1=nn.LayerNorm(d_model)
        self.norm2=nn.LayerNorm(d_model)
        self.FeedForward=FeedForward(d_model,n_neurons)

    def forward(self,X,mask):
        attention_output=self.RelativeAttentionBlock(X,X,X,mask)
        residual_output=X+attention_output
        normalized_residual_output=self.norm1(residual_output)
        feed_forward_output=self.FeedForward(normalized_residual_output)
        residual_feed_forward=normalized_residual_output+feed_forward_output
        normalized_feed_forward=self.norm2(residual_feed_forward)
        return normalized_feed_forward

In [7]:
#The Encoder block consists of n_layers as required for the task, the output of this block is used in the decoder block in cross_attention
class Encoder(nn.Module):
    def __init__(self,n_layers,d_model,n_qk,n_v,n_head,clipping_length,n_neurons):
        super().__init__()
        self.encoderlayers=nn.ModuleList([EncoderLayer(d_model,n_qk,n_v,n_head,clipping_length,n_neurons) for i in range(n_layers)])


    def forward(self,X,mask):
        for encoderlayer in self.encoderlayers:
            X=encoderlayer(X,mask)
        return X

In [8]:
#The decoder layer consists of a self relative position attention block with a residual connection and normalization, followed by a cross_attention block which uses the
#output of self attention block of this layer and the output of the Encoder block with a residual connection and normalization and finally the output of 
#block is fed into the fully connected layer with a residual connection and normalization
class DecoderLayer(nn.Module):
    def __init__(self,d_model,n_self_qk,n_self_v,n_self_head,clipping_length,n_cross_qk,n_cross_v,n_cross_head,n_neurons):
        super().__init__()
        self.RelativeSelfAttention=RelativeMultiHeadedAttention(d_model,n_self_qk,n_self_v,n_self_head,clipping_length)
        self.norm1=nn.LayerNorm(d_model)
        self.CrossAttention=MultiHeadedAttention(d_model,n_cross_qk,n_cross_v,n_cross_head)
        self.norm2=nn.LayerNorm(d_model)
        self.FeedForward=FeedForward(d_model,n_neurons)
        self.norm3=nn.LayerNorm(d_model)

    def forward(self,encoder_output,X,padding_mask,casual_mask):
        self_attention_output=self.RelativeSelfAttention(X,X,X,casual_mask)
        residual_self_attention=X+self_attention_output
        norm_residual_self=self.norm1(residual_self_attention)
        cross_attention_output=self.CrossAttention(Q_input=norm_residual_self,K_input=encoder_output,V_input=encoder_output,mask=padding_mask)
        residual_cross_attention=cross_attention_output+norm_residual_self
        norm_residual_cross=self.norm2(residual_cross_attention)
        feed_forward_output=self.FeedForward(norm_residual_cross)
        residual_feed_forward=feed_forward_output+norm_residual_cross
        norm_residual_feed=self.norm3(residual_feed_forward)
        return norm_residual_feed

In [9]:
#The Decoder Block consists of n_layers as required by the task, the output of this layer is sent for a linear layer for the prediction of next token
class Decoder(nn.Module):
    def __init__(self,n_layers,d_model,n_self_qk,n_self_v,n_self_head,clipping_length,n_cross_qk,n_cross_v,n_cross_head,n_neurons):
        super().__init__()
        self.decoderlayers=nn.ModuleList([DecoderLayer(d_model,n_self_qk,n_self_v,n_self_head,clipping_length,n_cross_qk,n_cross_v,n_cross_head,n_neurons) for i in range(n_layers)])

    def forward(self,encoder_output,X,padding_mask,casual_mask):
        for layer in self.decoderlayers:
            X=layer(encoder_output,X,padding_mask,casual_mask)
        return X

In [10]:
#The hyperparameters of the Transformer give a complete control over the number of layers,heads,neurons and dimensions of all the components and also on the clipping length
class Transformer(nn.Module):
    def __init__(self,d_model,clipping_length,n_seq,encoder_n_layers,decoder_n_layers,encoder_n_qk,encoder_n_v,encoder_n_head,decoder_n_self_qk,decoder_n_self_v,decoder_n_self_head,decoder_n_cross_qk,decoder_n_cross_v,decoder_n_cross_head,encoder_n_neurons,decoder_n_neurons):
        super().__init__()
        vocab_size=29 #Letters+start+end+pad
        self.src_embedding=nn.Embedding(vocab_size,d_model,padding_idx=28)   #Choosing the embeddings as:letters(0-25), start token(26),end token(27) and padding(28)
        self.target_embedding=nn.Embedding(vocab_size,d_model,padding_idx=28)
        self.Encoder=Encoder(encoder_n_layers,d_model,encoder_n_qk,encoder_n_v,encoder_n_head,clipping_length,encoder_n_neurons)#Encoder Block
        self.Decoder=Decoder(decoder_n_layers,d_model,decoder_n_self_qk,decoder_n_self_v,decoder_n_self_head,clipping_length,decoder_n_cross_qk,decoder_n_cross_v,decoder_n_cross_head,decoder_n_neurons)#Decoder Block
        self.output_logits=nn.Linear(d_model,vocab_size) #Logits which decide the next token
    def forward(self,X,target):
        padding_idx=28                                   #Setting the embedding of the pad token to 28
        encoder_padding_mask=(X==padding_idx)            #Mask of the Encoder block ignores all the padding tokens by making their attention scores zero
        embedded=self.src_embedding(X)                   #Embedding the input
        encoder_padding_mask=encoder_padding_mask.unsqueeze(1).unsqueeze(2) #Making it fit to be applied to the input X 
        encoded_output=self.Encoder(embedded,encoder_padding_mask)          #Pad Masking the Encoder Layer
        cross_attention_decoder_mask=(X==padding_idx)                       #Mask of the cross-attention layer in Decoder ignores all the padding tokens by making their attention scores zero
        cross_attention_decoder_mask=cross_attention_decoder_mask.unsqueeze(1).unsqueeze(2) #Making it fit to be applied to the output of self-attention layer
        self_padding_decoder_mask=(target==padding_idx).unsqueeze(1).unsqueeze(2)           #Padding mask and being fitted to be applied to the data label input   
        self_casual_decoder_mask=torch.triu(torch.ones(target.size(1),target.size(1)),diagonal=1).bool().to(target.device) #Making the map of the teacher forcing mask
        self_casual_decoder_mask=self_casual_decoder_mask.unsqueeze(0).unsqueeze(0)          #Making it fit to be applied to the data label input
        self_attention_decoder_mask=self_casual_decoder_mask|self_padding_decoder_mask       #Using the OR operation to combine the padding mask and the teacher forcing mask 
        target_embedded=self.target_embedding(target)                                        #Embedding the target input data
        decoded_output=self.Decoder(encoded_output,target_embedded,cross_attention_decoder_mask,self_attention_decoder_mask)#Decoder Block with its masks
        logits=self.output_logits(decoded_output) #Passing the decoder block output into a linear layer and getting logits for the prediction of the next token
        return logits                             #Gettings the logits as the output of the transformer

In [11]:
#Generating a dataset of sequences as the samples and the reversed sequences as the labels with the option to vary the n_samples and min_seq_len and 
#max_seq_len as per the need for trianing purposes as mentioned in the assignment we train it on random sequences with max_seq_len=50
from torch.utils.data import Dataset
import random
class SequenceDataset(Dataset):
    def __init__(self,min_letter,max_letter,min_seq_len,max_seq_len,n_samples):
        super().__init__()                     #Here letters are embedded as 0-25,start(26),end(27),pad(28)             
        self.dataset=[]                        #Dataset
        already_seen=set()                     #To prevent duplication keeping track of the data generated and it is a set to reduce computation time
        while len(self.dataset)<n_samples:     #To get n_samples in the dataset
            n_seq=random.randint(min_seq_len,max_seq_len-1) #Randomly initializing the seq_len
            data_sample=[random.randint(min_letter,max_letter) for i in range(n_seq)] #Producing the sample
            already_saw=tuple(data_sample)   #Making it into a tuple so as to store it in a set as a list cant be stored in a set
            if already_saw not in already_seen: #If unique
                already_seen.add(already_saw)   
                data_label=data_sample[::-1]    #Reversing the sample for producing the label 
                data_label_input=[26]+data_label.copy() #Adding the start token to the data_label input
                data_label_target=data_label.copy()+[27] #Adding the end token to the data_label target
                data_sample+=[28]*(max_seq_len-n_seq)    #Padding the sample
                data_label_input+=[28]*(max_seq_len-n_seq-1) #Padding data_label_input
                data_label_target+=[28]*(max_seq_len-n_seq-1)#Padding the data_label_target
                #Adding the data_sample,data_label_input and data_label_target into the dataset
                self.dataset.append((torch.tensor(data_sample),torch.tensor(data_label_input),torch.tensor(data_label_target))) 


    def __len__(self):
        return len(self.dataset)

    def __getitem__(self,idx):
        return self.dataset[idx]

In [12]:
df=SequenceDataset(min_letter=0,max_letter=25,min_seq_len=2,max_seq_len=51,n_samples=50000) #Initilalizing the dataset for trianing and testing with max_seq_len=50 
df_train,df_val,df_test=torch.utils.data.random_split(df,[35000,7500,7500])

In [13]:
from torch.utils.data import DataLoader
train_loader=DataLoader(df_train,batch_size=32,num_workers=4,shuffle=True)
val_loader=DataLoader(df_val,batch_size=32,num_workers=4)
test_loader=DataLoader(df_test,batch_size=32,num_workers=4)

In [14]:
#Keeping the Clipping Length as 10
model=Transformer(d_model=64,clipping_length=10,n_seq=51,encoder_n_layers=2,decoder_n_layers=2,encoder_n_qk=16,encoder_n_v=16,encoder_n_head=4,decoder_n_self_qk=16,decoder_n_self_v=16,decoder_n_self_head=4,decoder_n_cross_qk=16,decoder_n_cross_v=16,decoder_n_cross_head=4,encoder_n_neurons=128,decoder_n_neurons=128).to(device)
optimizer=torch.optim.Adam(model.parameters(),weight_decay=1e-4,betas=(0.9,0.999),lr=0.0001)
criterion=nn.CrossEntropyLoss(ignore_index=28)#Ignoring the paddings at time of CrossEntropyLoss calculation
learning_rate_scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode="max",patience=5,factor=0.5,min_lr=1e-6)

In [15]:
#Training with regularization,early stopping with patience of 11 and learning_rate_scheduling
def training(model,optimizer,criterion,train_loader,val_loader,learning_rate_scheduler,n_epochs):
    import copy
    best_val_acc=0
    patience=0
    for epoch in range(n_epochs):
        model.train()
        losses=0.0
        if patience>10: #Keeping a patience of 11
            break
        #Training the model and calculating the training loss
        for data_batch,label_input_batch,label_target_batch in train_loader:
            data_batch=data_batch.to(device)
            label_input_batch=label_input_batch.to(device)
            label_target_batch=label_target_batch.to(device)
            optimizer.zero_grad()
            y_probab=model(data_batch,label_input_batch)
            y_probab=y_probab.transpose(1,2)
            loss=criterion(y_probab,label_target_batch)
            losses+=loss.item()
            loss.backward()
            optimizer.step()
        training_loss=losses/len(train_loader)

        #Validation sequence accuracy evaluation and Learning rate scheduling
        model.eval()
        total_samples=0
        correct_sequence_reversal=0.0
        with torch.no_grad():
         for data_batch,label_input_batch,label_target_batch in val_loader:
            data_batch=data_batch.to(device)
            label_input_batch=label_input_batch.to(device)
            label_target_batch=label_target_batch.to(device)
            batch_size=data_batch.size(0)
            #In evaluation each input start with the Start token and keeps predicting the tokens one by one and add it to the input till End token is reached
            target=torch.full((batch_size,1),26,dtype=torch.long).to(device)
            for i in range(label_target_batch.size(1)):
                logits=model(data_batch,target)
                next_token=logits[:,-1,:].argmax(dim=-1,keepdim=True)
                target=torch.concat([target,next_token],dim=1)
            #Removing the start token as the labels don't contain them
            y_preds=target[:,1:]
            padding_mask=(label_target_batch!=28)
            #Accuracy is calculated on a sequence level and not token level thus if the sequence is reversed completely then only it will add to the accuracy
            correct_sequence_reversal+=((y_preds==label_target_batch)|(label_target_batch==28)).all(dim=1).sum().item()
            total_samples+=batch_size
        val_acc=correct_sequence_reversal/total_samples
        learning_rate_scheduler.step(val_acc)
        print(f"Epoch:{epoch},Loss:{training_loss},Val_accuracy:{val_acc:.5f}")

        #Early Stopping
        if val_acc>best_val_acc:
            best_val_acc=val_acc
            best_epoch=epoch
            best_model=copy.deepcopy(model.state_dict())
            patience=0 #We only track patience for consecutive non-repeating loops
        else:
            patience+=1
 
    #Saving the best model for evaluation purposes 
    torch.save(best_model,"/kaggle/working/bestmodel.pth")
    print(f"The best model is found at the epoch:{best_epoch} with a validation accuracy of {best_val_acc}")      

In [16]:
def eval(model,test_loader):
    #Calculating Test Accuracy
    model.eval()
    total_samples=0
    correct_sequence_reversal=0.0
    with torch.no_grad():
        for data_batch,label_input_batch,label_target_batch in test_loader:
            data_batch=data_batch.to(device)
            label_input_batch=label_input_batch.to(device)
            label_target_batch=label_target_batch.to(device)
            batch_size=data_batch.size(0)
            #In evaluation each input start with the Start token and keeps predicting the tokens one by one and add it to the input till End token is reached
            target=torch.full((batch_size,1),26,dtype=torch.long).to(device)
            for i in range(label_target_batch.size(1)):
                logits=model(data_batch,target)
                next_token=logits[:,-1,:].argmax(dim=-1,keepdim=True)
                target=torch.concat([target,next_token],dim=1)
            y_preds=target[:,1:]
            padding_mask=(label_target_batch!=28)
            #Accuracy is calculated on a sequence level and not token level thus if the sequence is reversed completely then only it will add to the accuracy
            correct_sequence_reversal+=((y_preds==label_target_batch)|(label_target_batch==28)).all(dim=1).sum().item()
            total_samples+=batch_size
    test_acc=correct_sequence_reversal/total_samples
    print(f"Test Accuracy:{test_acc}")

In [17]:
#Training with early stopping and 60 epochs
training(model,optimizer,criterion,train_loader,val_loader,learning_rate_scheduler,n_epochs=60)

Epoch:0,Loss:2.7221596722628996,Val_accuracy:0.03440
Epoch:1,Loss:0.7423687550006008,Val_accuracy:0.37133
Epoch:2,Loss:0.22179080846922053,Val_accuracy:0.65987
Epoch:3,Loss:0.10813412547002545,Val_accuracy:0.78000
Epoch:4,Loss:0.06597475308100854,Val_accuracy:0.85147
Epoch:5,Loss:0.05728172391765433,Val_accuracy:0.90400
Epoch:6,Loss:0.048438664435935114,Val_accuracy:0.92733
Epoch:7,Loss:0.01586131502297865,Val_accuracy:0.93920
Epoch:8,Loss:0.10953153180868502,Val_accuracy:0.95573
Epoch:9,Loss:0.05357050609270918,Val_accuracy:0.96133
Epoch:10,Loss:0.040280188973124735,Val_accuracy:0.95867
Epoch:11,Loss:0.030074969386950926,Val_accuracy:0.96667
Epoch:12,Loss:0.0066707573866836,Val_accuracy:0.97467
Epoch:13,Loss:0.10380460788735053,Val_accuracy:0.96360
Epoch:14,Loss:0.07796300768201415,Val_accuracy:0.30333
Epoch:15,Loss:0.010289624333488066,Val_accuracy:0.97107
Epoch:16,Loss:0.04676115166230325,Val_accuracy:0.98133
Epoch:17,Loss:0.049212201420767696,Val_accuracy:0.97987
Epoch:18,Loss:0.00

In [18]:
#Loading the best model found in early stopping and using it to evaluate the test accuracy
model.load_state_dict(torch.load('/kaggle/working/bestmodel.pth'))
eval(model,test_loader)

Test Accuracy:0.9993333333333333


In [19]:
#Seeing if the model generalizes well in sequences greater than 50 so using the best model to 
#evaluate the accuracy on a dataset with min_seq_len=51 and max_seq_len=72
df_test_2=SequenceDataset(min_letter=0,max_letter=25,min_seq_len=51,max_seq_len=72,n_samples=7500)
test_loader_2=DataLoader(df_test_2,batch_size=32,num_workers=4)
eval(model,test_loader_2)

Test Accuracy:0.282
